In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')

import torch
from torchsummary import summary
from thop import profile
import numpy as np
import time

import warnings
warnings.filterwarnings("ignore")

In [69]:
def measure_time_cpu(model, device, rep = 10):
    model = model.to(device=device)
    x = torch.rand((2, 16, 9), device=device)
    timings=np.zeros((rep,1))
    for i in range(rep):    
        start_time = time.time()
        out = model(x)
        timings[i] = time.time() - start_time
    mean_syn = np.sum(timings) / rep
    std_syn = np.std(timings)
    return mean_syn, std_syn

def measure_time_gpu(model, device, rep):
    model = model.to(device=device)
    dummy_input = torch.randn(2, 16, 9, dtype=torch.float).to(device)
    # INIT LOGGERS
    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    repetitions = rep
    timings=np.zeros((repetitions,1))
    #GPU-WARM-UP
    for _ in range(100):
        _ = model(dummy_input)
    # MEASURE PERFORMANCE
    with torch.no_grad():
        for rep in range(repetitions):
            starter.record()
            _ = model(dummy_input)
            ender.record()
            # WAIT FOR GPU SYNC
            torch.cuda.synchronize()
            curr_time = starter.elapsed_time(ender)
            timings[rep] = curr_time
    mean_syn = np.sum(timings) / repetitions
    std_syn = np.std(timings)
    return mean_syn, std_syn

In [3]:
from networks.conv_retnet import ConvRetNet

embedding_dim = 64
window_size = 32
cnn_1d_retnet_model = ConvRetNet(
                num_features=9,
                d_model=embedding_dim,
                out_channels_payload=embedding_dim,
                window_size=window_size,
                num_layers=2,
                num_heads=4,
                ffn_dim=embedding_dim*2,
                num_classes=10,
                dropout=0.1,
            ).to('cuda')

In [55]:
measure_time_gpu(cnn_1d_retnet_model, 'cuda', rep=1000)

(np.float64(3.610733694076538), np.float64(0.07701905876571062))

In [4]:
from thop import profile
wndow_size = 32
input_tensor = torch.randn(1, window_size, 9).to('cuda')
macs, params = profile(cnn_1d_retnet_model, inputs=(input_tensor,))
print(f"MACs: {f'{macs:,}' }, Parameters: {f'{params:,}' } using THOP")

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv1d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm1d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
MACs: 2,880,640.0, Parameters: 101,962.0 using THOP


In [14]:
from compare_networks.resconv_transformer import ResConvTransformer
from compare_networks.simple_cnn import SimpleCNNModel
from compare_networks.stdlm import STDLM, STDLMConfig
from compare_networks.transformer_baseline import TransformerBaselineModel
from compare_networks.ae_transformer import AETransformer, compute_timestamp_encoding


cfg = STDLMConfig(window_size=32, num_classes=10)
stdlm_model = STDLM(cfg).to('cuda')

simple_cnn_model = SimpleCNNModel(
            num_features=9, d_model=128, num_layers=4,
            kernel_size=3, num_classes=10, dropout=0.1,
).to('cuda')
transformer_model = TransformerBaselineModel(
            num_features=9, d_model=128, num_heads=4, ff_dim=256,
            num_layers=2, num_classes=10, dropout=0.1,
    ).to('cuda')
rescnn_transformer = ResConvTransformer(
            num_features=9, 
            d_model=128, 
            out_channels_payload=128, 
            window_size=32, 
            num_layers=2, 
            num_heads=4, 
            ffn_dim=256, 
            num_classes=10, 
            dropout=0.1,
    ).to('cuda')

ae_transformer = AETransformer(
        mes_dim=12, dout_mess=10,
        num_heads=10, num_layers=6,
        num_classes=10, tse=True, dropout=0.1,
    ).to('cuda')


In [15]:
input_tensor = torch.randn(1, 32, 9).to('cuda')

In [16]:
print(f"Simple CNN Model:")
macs, params = profile(simple_cnn_model, inputs=(input_tensor,))
print(f"MACs: {f'{macs:,}' }, Parameters: {f'{params:,}' } using THOP")

Simple CNN Model:
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv1d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm1d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
MACs: 4,962,560.0, Parameters: 203,914.0 using THOP


In [35]:
measure_time_gpu(simple_cnn_model, 'cuda', rep=1000)

(np.float64(1.1727585991621017), np.float64(0.06042027151033132))

In [17]:
print(f"Transformer Model:")
macs, params = profile(transformer_model, inputs=(input_tensor,))
print(f"MACs: {f'{macs:,}' }, Parameters: {f'{params:,}' } using THOP")


Transformer Model:
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
MACs: 8,558,848.0, Parameters: 317,706.0 using THOP


In [57]:
measure_time_gpu(transformer_model, 'cuda', rep=1000)

(np.float64(1.6546786243915559), np.float64(0.036343059570839914))

In [18]:

print(f"ResConvTransformer Model:")
macs, params = profile(rescnn_transformer, inputs=(input_tensor,))
print(f"MACs: {f'{macs:,}' }, Parameters: {f'{params:,}' } using THOP")



ResConvTransformer Model:
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv1d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm1d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
MACs: 10,250,496.0, Parameters: 369,802.0 using THOP


In [58]:
measure_time_gpu(rescnn_transformer, 'cuda', rep=1000)

(np.float64(2.5477774436473846), np.float64(0.09239103514220384))

In [19]:
print(f"STDLM Model:")
macs, params = profile(stdlm_model, inputs=(input_tensor,))
print(f"MACs: {f'{macs:,}' }, Parameters: {f'{params:,}' } using THOP")


STDLM Model:
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv1d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm1d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool1d'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
MACs: 5,584,192.0, Parameters: 174,318.0 using THOP


In [59]:
measure_time_gpu(stdlm_model, 'cuda', rep=1000)

(np.float64(6.589494054317474), np.float64(0.41584088873381037))

In [27]:
T = 32  # <-- set to the window length from the .npy files
mes = torch.randn(1, T, 12, device='cuda')
ts = torch.rand(1, T, device='cuda').sort(dim=1).values  # increasing timestamps


In [34]:
import torch.nn as nn

class AETransformerInference(nn.Module):
    def __init__(self, model, gran=1e-7):
        super().__init__()
        self.model, self.gran = model, gran

    def forward(self, mes, raw_ts):
        tp = compute_timestamp_encoding(raw_ts, d_model=self.model.d_model, gran=self.gran)
        return self.model(mes, time_position=tp)

wrapped = AETransformerInference(ae_transformer).to('cuda').eval()

T = 32
mes = torch.randn(1, T, 12, device='cuda')
# raw_ts = torch.cumsum(torch.rand(1, T, device='cuda') * 1e-3, dim=1)  # increasing timestamps
raw_ts = torch.rand(1, T, device='cuda')  # increasing timestamps

macs, params = profile(wrapped, inputs=(mes, raw_ts), verbose=False)
print(f"MACs: {f'{macs:,}' }, Parameters: {f'{params:,}' } using THOP")
# with torch.inference_mode():
#     out = wrapped(mes, raw_ts)

MACs: 55,395,004.0, Parameters: 259,572.0 using THOP


In [61]:
import numpy as np
import torch
from compare_networks.ae_transformer import (
    AETransformer, TimestampPositionTable, discretize_timestamp,
)


def measure_time_gpu_ae(model, device, rep, batch_size=2, window_size=32,
                        gran=1e-7, max_time_position=10000):
    """GPU latency of AE-Transformer, including the timestamp encoding.

    Returns (mean_ms, std_ms) per forward pass of `batch_size` windows.
    """
    model = model.to(device=device).eval()

    # Same input format as the dataset: (B, T, 12) messages + (B, T) raw timestamps
    mes = torch.randn(batch_size, window_size, 12, dtype=torch.float, device=device)
    raw_ts = torch.cumsum(torch.rand(batch_size, window_size, device=device) * 1e-3, dim=1)

    # Build the sinusoidal lookup table ONCE (as in the original implementation)
    table = TimestampPositionTable(embed=model.d_model,
                                   max_time_position=max_time_position).to(device)

    def infer():
        idx = discretize_timestamp(raw_ts, gran=gran)      # (B, T) indices
        time_position = table.lookup(idx)                  # (B, T, d_model)
        return model(mes, time_position=time_position)

    # INIT LOGGERS
    starter = torch.cuda.Event(enable_timing=True)
    ender = torch.cuda.Event(enable_timing=True)
    timings = np.zeros((rep, 1))

    with torch.no_grad():
        # GPU WARM-UP
        for _ in range(100):
            _ = infer()
        torch.cuda.synchronize()

        # MEASURE PERFORMANCE
        for r in range(rep):
            starter.record()
            _ = infer()
            ender.record()
            torch.cuda.synchronize()          # WAIT FOR GPU SYNC
            timings[r] = starter.elapsed_time(ender)

    return np.mean(timings), np.std(timings)


# Example
model = AETransformer(dout_mess=10, num_heads=10, num_layers=6,
                      num_classes=10, tse=True)
measure_time_gpu_ae(model, 'cuda', rep=1000)

(np.float64(2.424252093553543), np.float64(0.04420135792678788))

In [89]:
import io
import gc
import torch

MB = 1024 ** 2


def model_size(model):
    """Static size of parameters and buffers held in memory (MB)."""
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
    return {
        "params": sum(p.numel() for p in model.parameters()),
        "param_MB": param_bytes / MB,
        "buffer_MB": buffer_bytes / MB,
        "total_MB": (param_bytes + buffer_bytes) / MB,
    }


def disk_size(model):
    """Size of the saved state_dict (MB), i.e. the checkpoint file size."""
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.getbuffer().nbytes / MB


def peak_gpu_memory(model, inputs, device="cuda", rep=10):
    """Peak GPU memory during inference (MB).

    inputs: a tensor or a tuple of tensors, e.g. (mes, raw_ts) for AE-Transformer.
    Returns the total peak (weights + activations) and the activation part only.
    """
    if not isinstance(inputs, (tuple, list)):
        inputs = (inputs,)
    model = model.to(device).eval()
    inputs = tuple(x.to(device) for x in inputs)

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats(device)
    base = torch.cuda.memory_allocated(device)          # weights + inputs already on GPU

    with torch.no_grad():
        for _ in range(rep):
            model(*inputs)
    torch.cuda.synchronize()

    peak = torch.cuda.max_memory_allocated(device)
    return {"peak_total_MB": peak / MB,
            "peak_activation_MB": (peak - base) / MB}


def report(model, inputs, name="model", device="cuda"):
    s = model_size(model)
    print(f"===== {name} =====")
    print(f"Parameters            : {s['params']:,}")
    print(f"Parameter memory      : {s['param_MB']:.4f} MB")
    print(f"Buffer memory         : {s['buffer_MB']:.4f} MB")
    print(f"Static model size     : {s['total_MB']:.4f} MB")
    print(f"Checkpoint size (disk): {disk_size(model):.4f} MB")
    if device == "cuda" and torch.cuda.is_available():
        g = peak_gpu_memory(model, inputs, device)
        print(f"Peak GPU memory       : {g['peak_total_MB']:.4f} MB")
        print(f"  of which activations: {g['peak_activation_MB']:.4f} MB")


In [93]:
report(cnn_1d_retnet_model, torch.randn(1, 64, 9), name="Proposed")

===== Proposed =====
Parameters            : 401,034
Parameter memory      : 1.5298 MB
Buffer memory         : 0.0063 MB
Static model size     : 1.5361 MB
Checkpoint size (disk): 1.5751 MB
Peak GPU memory       : 17.2725 MB
  of which activations: 1.0342 MB
